In [1]:
from neuromodes.mbm import mbm_calc_p_vals
import numpy as np
from scipy.stats import genpareto

c = .1
loc = 1
scale = 1


ptrue = 0.01
nnull = 10000

# get the centile element from the gpd
obs = genpareto.ppf(1 - ptrue, c, loc=loc, scale=scale)
print(obs)

# generate nulls from the gpd
nulls = genpareto.rvs(c, loc=loc, scale=scale, size=nnull)

# naive p-value
pnaive = np.mean(nulls >= obs)

# fit and get p-value
fit = genpareto.fit(nulls, floc=loc)
pfitted = genpareto.sf(obs, *fit)

print(f"Naive error:  {np.abs(pnaive - ptrue)}")
print(f"Fitted error: {np.abs(pfitted - ptrue)}")



6.848931924611134
Naive error:  0.00020000000000000052
Fitted error: 0.00018315524081052338


In [16]:
import numpy as np
from scipy.stats import genpareto
from neuromodes.mbm import mbm_calc_p_vals

c, loc, scale = 0.1, 1.0, 1.0
true_p = 0.01
obs = genpareto.ppf(1-true_p, c, loc=loc, scale=scale)

n_sims = 10
nnull = 100
ntail = 25

naive_errors = []
fitted1_errors = []
fitted2_errors = []

for _ in range(n_sims):
    nulls = np.sort(genpareto.rvs(c, loc=loc, scale=scale, size=(1,nnull)))

    pnaive = np.mean(nulls >= obs)

    # Fixing loc=loc if known gives maximum parametric efficiency,
    # but even unconstrained fit beats empirical count variance on average
    floc = np.mean(nulls[:,-ntail-1:-ntail+1])
    fit = genpareto.fit(nulls[:,-ntail:], floc = floc)
    pfitted1 = genpareto.sf(obs, *fit) * ntail/nnull

    pfitted2 = mbm_calc_p_vals(np.array([obs]), 
                              nulls,
                              alternative='greater', 
                              n_for_tail_estimation=(ntail,),
                              n_for_pval_recalculation=np.inf, 
                              )[0]

    np.testing.assert_allclose(pfitted1, pfitted2, rtol=1e-9, atol=1e-9)
    naive_errors.append((pnaive - true_p) ** 2)
    fitted1_errors.append((pfitted1 - true_p) ** 2)
    fitted2_errors.append((pfitted2 - true_p) ** 2)

print(f"MSE Naive:   {np.mean(naive_errors):.6f}")
print(f"MSE Fitted1: {np.mean(fitted1_errors):.6f}")
print(f"MSE Fitted2: {np.mean(fitted2_errors):.6f}")

MSE Naive:   0.000090
MSE Fitted1: 0.000097
MSE Fitted2: 0.000097
